# RavenPack Ingest Verification

Compares 5 random samples per month between:
- **Archive** (`derived/.archive/headline_embeddings/`) — old embed_headlines output, contains RP_STORY_ID + HEADLINE
- **New ingest** (`derived/ravenpack_headlines/.../`) — new pipeline output

Checks that sampled `RP_STORY_ID` and `HEADLINE` values match exactly.

In [1]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

DATALAKE_ROOT = Path(os.environ["DATALAKE_ROOT"])
ARCHIVE_DIR   = DATALAKE_ROOT / "derived" / ".archive" / "headline_embeddings"
INGEST_DIR    = sorted(
    (DATALAKE_ROOT / "derived" / "ravenpack_headlines").glob(
        "ravenpack_headlines__v0.1.0__*"
    )
)[0]

# print(f"Archive : {ARCHIVE_DIR}")
# print(f"Ingest  : {INGEST_DIR}")

archive_months = sorted(ARCHIVE_DIR.glob("*.parquet"))
ingest_months  = sorted(INGEST_DIR.glob("*.parquet"))
print(f"\nArchive months : {len(archive_months)}")
print(f"Ingest months  : {len(ingest_months)}")



Archive months : 0
Ingest months  : 312


In [3]:
import polars as pl
import random

random.seed(42)

SAMPLE_N = 5
results = []

for archive_path in archive_months:
    month = archive_path.name  # e.g. 2000-01.parquet
    ingest_path = INGEST_DIR / month
    if not ingest_path.exists():
        results.append({"month": month, "status": "MISSING_IN_INGEST",
                        "n_archive": None, "n_ingest": None,
                        "n_matched": None, "mismatches": []})
        continue

    # Load only needed columns -- embedding column is huge, skip it
    archive_cols = pl.read_parquet(archive_path).select(["RP_STORY_ID", "HEADLINE"])
    ingest_cols  = pl.read_parquet(ingest_path).select(["RP_STORY_ID", "HEADLINE"])

    # Sample 5 random rows from the archive
    n = len(archive_cols)
    sample_idx = random.sample(range(n), min(SAMPLE_N, n))
    archive_sample = archive_cols[sample_idx]

    # Look them up in the new ingest by RP_STORY_ID
    story_ids = archive_sample["RP_STORY_ID"].to_list()
    ingest_lookup = (
        ingest_cols
        .filter(pl.col("RP_STORY_ID").is_in(story_ids))
        .sort("RP_STORY_ID")
    )
    archive_lookup = archive_sample.sort("RP_STORY_ID")

    mismatches = []
    for row_a, row_i in zip(archive_lookup.iter_rows(named=True),
                             ingest_lookup.iter_rows(named=True)):
        if row_a["RP_STORY_ID"] != row_i["RP_STORY_ID"]:
            mismatches.append({"story_id": row_a["RP_STORY_ID"],
                                "issue": "story_id_mismatch"})
        elif row_a["HEADLINE"] != row_i["HEADLINE"]:
            mismatches.append({
                "story_id": row_a["RP_STORY_ID"],
                "issue": "headline_mismatch",
                "archive":  row_a["HEADLINE"],
                "ingest":   row_i["HEADLINE"],
            })

    found = len(ingest_lookup)

    status_str = "OK" if not mismatches and found == len(story_ids) else "MISMATCH"
    print(f"{month}: {status_str} | archive={n:,} ingest={len(ingest_cols):,} sampled={len(story_ids)} found={found}")
    results.append({
        "month":      month,
        "status":     "OK" if not mismatches and found == len(story_ids) else "MISMATCH",
        "n_archive":  n,
        "n_ingest":   len(ingest_cols),
        "n_sampled":  len(story_ids),
        "n_found":    found,
        "mismatches": mismatches,
    })

print(f"Checked {len(results)} months")


2000-01.parquet: OK | archive=105,404 ingest=105,404 sampled=5 found=5
2000-02.parquet: OK | archive=165,503 ingest=165,503 sampled=5 found=5
2000-03.parquet: OK | archive=97,536 ingest=97,536 sampled=5 found=5
2000-04.parquet: OK | archive=76,781 ingest=76,781 sampled=5 found=5
2000-05.parquet: OK | archive=168,845 ingest=168,845 sampled=5 found=5
2000-06.parquet: OK | archive=175,763 ingest=175,763 sampled=5 found=5
2000-07.parquet: OK | archive=176,871 ingest=176,871 sampled=5 found=5
2000-08.parquet: OK | archive=184,189 ingest=184,189 sampled=5 found=5
2000-09.parquet: OK | archive=160,963 ingest=160,963 sampled=5 found=5
2000-10.parquet: OK | archive=85,430 ingest=85,430 sampled=5 found=5
2000-11.parquet: OK | archive=88,150 ingest=88,150 sampled=5 found=5
2000-12.parquet: OK | archive=75,131 ingest=75,131 sampled=5 found=5
2001-01.parquet: OK | archive=185,564 ingest=185,564 sampled=5 found=5
2001-02.parquet: OK | archive=194,528 ingest=194,528 sampled=5 found=5
2001-03.parquet:

In [4]:
# Summary table
summary = pl.DataFrame([
    {
        "month":     r["month"],
        "status":    r["status"],
        "n_archive": r["n_archive"],
        "n_ingest":  r["n_ingest"],
        "n_sampled": r.get("n_sampled"),
        "n_found":   r.get("n_found"),
        "n_mismatches": len(r["mismatches"]),
    }
    for r in results
])

print(summary.filter(pl.col("status") != "OK"))
print(f"\n{summary['status'].value_counts(sort=True)}")


shape: (0, 7)
┌───────┬────────┬───────────┬──────────┬───────────┬─────────┬──────────────┐
│ month ┆ status ┆ n_archive ┆ n_ingest ┆ n_sampled ┆ n_found ┆ n_mismatches │
│ ---   ┆ ---    ┆ ---       ┆ ---      ┆ ---       ┆ ---     ┆ ---          │
│ str   ┆ str    ┆ i64       ┆ i64      ┆ i64       ┆ i64     ┆ i64          │
╞═══════╪════════╪═══════════╪══════════╪═══════════╪═════════╪══════════════╡
└───────┴────────┴───────────┴──────────┴───────────┴─────────┴──────────────┘

shape: (1, 2)
┌────────┬───────┐
│ status ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ OK     ┆ 312   │
└────────┴───────┘


In [5]:
# Detail mismatches if any
for r in results:
    if r["mismatches"]:
        print(f"\n=== {r['month']} ===")
        for m in r["mismatches"]:
            print(m)

if all(r["status"] == "OK" for r in results):
    print("ALL OK -- new ingest matches archive for all sampled story IDs and headlines.")


ALL OK -- new ingest matches archive for all sampled story IDs and headlines.


In [6]:
# Row count comparison across all months
count_df = pl.DataFrame([
    {"month": r["month"], "n_archive": r["n_archive"], "n_ingest": r["n_ingest"]}
    for r in results
    if r["n_archive"] is not None
]).with_columns(
    (pl.col("n_ingest") - pl.col("n_archive")).alias("delta")
)

print("Months where row counts differ:")
print(count_df.filter(pl.col("delta") != 0))
print(f"\nTotal archive stories : {count_df['n_archive'].sum():,}")
print(f"Total ingest stories  : {count_df['n_ingest'].sum():,}")
print(f"Delta                 : {count_df['delta'].sum():,}")


Months where row counts differ:
shape: (0, 4)
┌───────┬───────────┬──────────┬───────┐
│ month ┆ n_archive ┆ n_ingest ┆ delta │
│ ---   ┆ ---       ┆ ---      ┆ ---   │
│ str   ┆ i64       ┆ i64      ┆ i64   │
╞═══════╪═══════════╪══════════╪═══════╡
└───────┴───────────┴──────────┴───────┘

Total archive stories : 997,066,086
Total ingest stories  : 997,066,086
Delta                 : 0
